# 1 — Modular Multilevel Converter (MMC) Modeling

> **Goal.** Derive the MMC from first principles: the half-bridge
> sub-module, the arm KVL/KCL equations, the **circulating current**
> that distinguishes MMC from any simpler topology, and the
> phase-shifted carrier (PSC) PWM modulator. By the end you'll
> understand why MMC scales to hundreds of voltage levels and what
> control challenges that creates.

**Prerequisites**

- The 2-level VSI project (`../vsi_3phase/`) — we reuse dq-frame
  control intuition.
- The NPC 3-level project (`../npc_3phase/`) — multilevel basics +
  capacitor balancing as a control challenge.

**What you'll be able to do at the end**

1. Draw the half-bridge sub-module (HB-SM) and state its two valid
   operating modes (insert / bypass).
2. Write the arm KVL: $v_{arm}(t)$ is the sum of inserted SM
   capacitor voltages.
3. Write the loop KVL across the bus: derive the **circulating
   current** equation as the difference between expected DC
   $V_{dc}/2$ and the actual sum $v_{arm,upper} + v_{arm,lower}$.
4. Build a PSC-PWM modulator with $N$ phase-shifted carriers and show
   the effective load-side ripple is at $N f_{carrier}$.
5. Compute the **steady-state cap voltage** $V_C = V_{dc}/N$ from
   the arm energy balance.
6. Sketch the 3-phase extension and state how 3 phases share the bus.


## Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import matplotlib.pyplot as plt

from mmc_model import (
    MMCParams,
    psc_pwm_insertion_count_vectorised,
    arm_references,
    decompose_arm_currents,
    operating_point_report,
)

plt.rcParams["figure.figsize"] = (11, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

p = MMCParams()
print(operating_point_report(p))


## 1. The half-bridge sub-module (HB-SM)

The MMC building block is a tiny half-bridge converter with **two
IGBTs** ($S_1$ and $S_2$) and **one capacitor** ($C_{SM}$):

```
       Terminal A (midpoint of S1, S2)
            │
       ┌────┤
       │    │
       │   S1
       │    │
     C_SM   midpoint
       │    │
       │   S2
       │    │
       └────┤
            │
       Terminal B
```

Two valid states:

| State | $S_1$ | $S_2$ | $v_{SM} = v_A - v_B$ |
|:-:|:-:|:-:|:-:|
| **INSERT** | ON | OFF | $+V_C$ (cap in series with arm current) |
| **BYPASS** | OFF | ON | $0$ (cap isolated, current bypasses) |

Either-both states are forbidden (short the cap or float the
arm). The interesting thing: **in INSERT mode the arm current
charges or discharges the cap**, depending on its sign. In BYPASS
the cap is held — no current flows through it. So $V_C$ is a state
variable controlled by *when* we insert this SM relative to the arm
current's sign.

For $N$ SMs in an arm, the arm voltage is

$$
v_{arm}(t) = \sum_{i=1}^{N} s_i(t) \cdot v_{C,i}(t)
$$

where $s_i(t) \in \{0, 1\}$ is the insertion flag for SM $i$.
If all caps are at $V_C$, then $v_{arm} = n(t) \cdot V_C$ where
$n(t) = \sum s_i$ takes values in $\{0, 1, \ldots, N\}$ — **N+1
discrete levels per arm**.


## 2. Steady-state cap voltage: $V_C = V_{dc}/N$

In DC steady-state the arm voltages must sum to the bus voltage
(KVL through the two arms, neglecting the arm inductors at DC):

$$
\overline{v_{arm,upper}} + \overline{v_{arm,lower}} = V_{dc}
$$

For a symmetric inverter, $\overline{n_{upper}} = \overline{n_{lower}}
= N/2$ on average. With $\overline{v_{arm}} = N V_C / 2$ on each
side, the KVL becomes

$$
\frac{N V_C}{2} + \frac{N V_C}{2} = V_{dc} \implies \boxed{V_C = \frac{V_{dc}}{N}}
$$

That's why the MMC's per-SM voltage rating scales as $V_{dc}/N$ —
the headline benefit of multilevel: **double N → halve the per-SM
voltage stress**.


## 3. The arm KVL — and the circulating current

KVL around the loop spanning both arms + the bus:

$$
+\frac{V_{dc}}{2} - L_{arm} \frac{di_{arm,up}}{dt} - v_{arm,up}(t)
- v_{ac}(t)
+ v_{arm,lo}(t) + L_{arm} \frac{di_{arm,lo}}{dt}
- \left(-\frac{V_{dc}}{2}\right) = 0
$$

Define the **circulating current**:

$$
i_{circ}(t) = \frac{i_{arm,up}(t) + i_{arm,lo}(t)}{2}
$$

and the **AC current**:

$$
i_{ac}(t) = i_{arm,up}(t) - i_{arm,lo}(t)
$$

(by KCL at the AC node). Substituting and simplifying:

$$
2 L_{arm} \frac{d i_{circ}}{dt} = V_{dc} - (v_{arm,up} + v_{arm,lo})
$$

$$
v_{ac}(t) = \frac{V_{dc}}{2} - v_{arm,up}(t)
          = -\frac{V_{dc}}{2} + v_{arm,lo}(t)
$$

The **circulating current** is driven by any imbalance between
$(v_{arm,up} + v_{arm,lo})$ and $V_{dc}$. In steady state with caps
balanced, the sum equals $V_{dc}$ on average — but there's a
**$2 f_o$ ripple** from the modulation that pumps $i_{circ}$ at twice
the line frequency. That parasitic component must be suppressed by a
**resonant controller** (covered in
[`02_mmc_control.ipynb`](02_mmc_control.ipynb)).


## 4. Phase-Shifted Carrier (PSC) PWM

With $N$ SMs per arm, we use $N$ triangular carriers all at
$f_{carrier}$ but **phase-shifted by $2\pi / N$** from each other.
The reference for the upper arm is

$$
d_{up}(t) = \frac{1 - m_a \sin(\omega_o t)}{2} \in [0, 1]
$$

(the lower arm is the complement, $d_{lo} = 1 - d_{up}$ in the
balanced case). At each instant, count how many of the $N$ carriers
are **below** $d_{up}(t)$ — that's the **number of SMs to insert**.

This scheme has two beautiful properties:

1. **The N carriers' ripples cancel symmetrically** → the effective
   ripple in the output voltage appears at $N \cdot f_{carrier}$
   instead of $f_{carrier}$. So the load sees a much cleaner output.
2. **Each SM switches at exactly $f_{carrier}$** → switching losses
   per SM are independent of $N$. Total losses scale with $N$ (more
   SMs) but the average doesn't get worse.


In [ ]:
# Visualise PSC-PWM for the upper arm over one fundamental period.
t = np.arange(0, 1.0/p.f_o, 1/(p.f_carrier * 200))
d_up, d_lo = arm_references(p, t)

n_up = psc_pwm_insertion_count_vectorised(d_up, t, p.f_carrier, p.N_sm)
v_arm_up_ideal = n_up * p.V_C_nominal
v_ac_ideal = p.V_dc_half - v_arm_up_ideal

fig, (ax_carriers, ax_arm, ax_ac) = plt.subplots(3, 1, figsize=(11, 9), sharex=True)

# Top: reference + N phase-shifted carriers.
ax_carriers.plot(t * 1e3, d_up, color="C3", lw=1.6, label=fr"$d_{{up}}(t)$, $m_a={p.m_a:.2f}$")
for k in range(p.N_sm):
    phi = (2*np.pi*p.f_carrier*t + k*2*np.pi/p.N_sm) % (2*np.pi)
    tri = 0.5 + 0.5 * (2/np.pi) * np.arcsin(np.sin(phi))
    ax_carriers.plot(t * 1e3, tri, lw=0.5, alpha=0.7,
                       label=f"carrier {k}")
ax_carriers.set_ylabel("Normalised reference / carriers")
ax_carriers.set_title(f"PSC-PWM upper-arm: 1 reference + {p.N_sm} phase-shifted carriers")
ax_carriers.legend(loc="upper right", fontsize=8, ncol=2)

# Middle: arm voltage = n × V_C in 4 discrete levels.
ax_arm.plot(t * 1e3, v_arm_up_ideal, color="C0", lw=0.8)
for k in range(p.N_sm + 1):
    ax_arm.axhline(k * p.V_C_nominal, color="k", ls=":", alpha=0.25)
ax_arm.set_ylabel("$v_{arm,up}$ [V]")
ax_arm.set_title(f"Upper-arm voltage — {p.N_sm+1} levels at "
                  f"{p.V_C_nominal:.1f} V spacing")

# Bottom: AC output = V_dc/2 - v_arm_up
ax_ac.plot(t * 1e3, v_ac_ideal, color="C2", lw=0.8)
omega = 2*np.pi*p.f_o
v_ac_fund = p.V_o_pk * np.sin(omega * t)
ax_ac.plot(t * 1e3, v_ac_fund, color="C3", lw=1.5,
            label=fr"Analytical fundamental, $V_{{o,pk}}={p.V_o_pk:.1f}$ V")
ax_ac.set_xlabel("Time [ms]")
ax_ac.set_ylabel("$v_{ac}$ [V]")
ax_ac.legend(loc="lower right")

plt.tight_layout()
plt.show()


## 5. The circulating current at $2 f_o$

The arm voltages contain a $2 f_o$ component because each arm
voltage $\overline{v_{arm}}(t) = \frac{V_{dc}}{2}(1 \mp m_a \sin \omega_o t)$
times the load current modulates the power flow at twice the line
frequency. When the cap voltages "breathe" with this $2 f_o$ envelope,
the sum $v_{arm,up} + v_{arm,lo}$ no longer equals $V_{dc}$ exactly,
which drives the circulating-current dynamics:

$$
2 L_{arm} \frac{d i_{circ}}{dt} = V_{dc} - (v_{arm,up} + v_{arm,lo})
\approx -\Delta v_{2 f_o} \cdot \sin(2 \omega_o t + \phi)
$$

For our parameters, the $2 f_o$ circulating current amplitude can
reach a noticeable fraction of $I_o$ if left unsuppressed. The
notebook [`02_mmc_control.ipynb`](02_mmc_control.ipynb) implements a
**resonant controller** centered at $2 f_o$ to drive this component
toward zero.


## 6. 3-phase extension

A 3-phase MMC is simply **three identical phase legs** sharing the
DC bus:

```
   +V_dc/2 ─┬───────┬───────┬───
            │       │       │
           leg     leg     leg
           A       B       C
            │       │       │
   -V_dc/2 ─┴───────┴───────┴───
            │       │       │
            ac_a   ac_b   ac_c   →  load / grid
```

Each leg has 2 arms × $N$ SMs = $2N$ switches × 3 phases = $6N$
controllable switches in total. For HVDC with $N = 300$ that's
1800 switches per converter — and the topology generalises to even
higher numbers.

The 3 legs operate independently for AC modulation (each leg's
$v_{ac,\phi}$ is set by its own PSC-PWM with the phase shift) but
**share the DC bus** so the per-cap currents from all 3 phases sum
in the bus capacitor.

For balanced 3-phase operation, the circulating $2 f_o$ ripple
cancels across the 3 legs (zero-sequence), so the DC-side current
is much cleaner than a 2-level VSI's.

In this project we model **single-phase MMC** to keep the Pulsim
simulation tractable. The control machinery (sort-and-select +
resonant suppression + dq) is unchanged from 1-phase to 3-phase.


## Summary

- The **MMC** is a stack of identical sub-modules per arm, with two
  arms per phase. Each SM is a half-bridge with two switches and one
  capacitor; INSERT / BYPASS gives $v_{SM} = V_C$ or $0$.
- The arm voltage is the sum of inserted SM cap voltages →
  $N+1$ discrete levels per arm in DC steady state.
- The **arm KVL** yields the **circulating current** equation,
  driven by any imbalance between $V_{dc}$ and $v_{arm,up} + v_{arm,lo}$.
- **PSC-PWM** with $N$ phase-shifted carriers gives effective
  load-side ripple at $N f_{carrier}$ — the multilevel
  switching-frequency dividend.
- **Steady-state cap voltage** $V_C = V_{dc}/N$, set by arm energy
  balance.

**Cross-validation against Pulsim**: open
[`00_mmc_pulsim_validation.ipynb`](00_mmc_pulsim_validation.ipynb)
for an executed single-phase MMC simulation showing the 4-level
arm voltage + 4-level AC output with all SM caps locked to
$V_C$ via sort-and-select balancing.

**Next**: [`02_mmc_control.ipynb`](02_mmc_control.ipynb) covers the
two MMC-specific controllers: sort-and-select capacitor balancing
and the $2 f_o$ resonant suppressor for the circulating current.

**Suggested exercises**

1. Compute $V_C = V_{dc}/N$ for $N = 10, 20, 100$. How does the
   per-SM switch voltage stress scale?
2. Show that adding a $3rd$-harmonic to the reference $d_{up}(t)$
   doesn't change $v_{ac}$ (3rd-harmonic injection cancels in
   line-to-line voltages in a 3-phase system — same idea as the
   2-level VSI's SVPWM trick).
3. Derive the **full-bridge sub-module (FB-SM)** voltage range —
   it's $\{+V_C, 0, -V_C\}$ instead of $\{+V_C, 0\}$. How does that
   change the MMC's voltage range capability?
